In [1]:

from pathlib import Path

import joblib
import pandas as pd

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


# --------------------------------------------------
# Load trained model
# --------------------------------------------------

BASE_DIR = Path(__file__).resolve().parent.parent
MODEL_PATH = BASE_DIR / "models" / "model.pkl"

model = joblib.load(MODEL_PATH)


# --------------------------------------------------
# Define FastAPI application
# --------------------------------------------------

app = FastAPI(
    title="Retail Campaign Response API",
    description="API for predicting customer campaign response.",
    version="1.0.0"
)


# --------------------------------------------------
# Model input features
# Must match model training features
# --------------------------------------------------

FEATURES = [
    "n_comp",
    "loyalty",
    "nps",
    "n_communications",
    "total_sales",
    "unique_products",
    "number_of_invoices",
    "purchase_days",
    "average_invoice_value",
    "average_quantity_per_invoice",
    "average_product_price",
    "customer_activity_days"
]


# --------------------------------------------------
# Input validation schema
# --------------------------------------------------

class CustomerInput(BaseModel):

    n_comp: float
    loyalty: int = Field(ge=0, le=1)
    nps: float = Field(ge=0, le=10)
    n_communications: float = Field(ge=0)

    total_sales: float
    unique_products: float = Field(ge=0)
    number_of_invoices: float = Field(ge=0)
    purchase_days: float = Field(ge=0)

    average_invoice_value: float
    average_quantity_per_invoice: float
    average_product_price: float
    customer_activity_days: float = Field(ge=0)


# --------------------------------------------------
# API home endpoint
# --------------------------------------------------

@app.get("/")
def home():

    return {
        "message": "Retail Campaign Response API is running",
        "docs": "/docs"
    }


# --------------------------------------------------
# Prediction endpoint
# --------------------------------------------------

@app.post("/predict")
def predict(customer: CustomerInput):

    try:

        # Convert validated input to DataFrame
        customer_data = pd.DataFrame(
            [customer.model_dump()]
        )

        # Ensure correct feature order
        customer_data = customer_data[FEATURES]

        # Generate probability and prediction
        probability = float(
            model.predict_proba(customer_data)[0][1]
        )

        prediction = int(
            model.predict(customer_data)[0]
        )

        return {
            "predicted_response": prediction,
            "prediction_label": (
                "Response" if prediction == 1
                else "No Response"
            ),
            "response_probability": round(probability, 4),
            "response_probability_percentage": round(
                probability * 100, 2
            )
        }

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=f"Prediction failed: {str(e)}"
        )

NameError: name '__file__' is not defined